# VM2 — Can: td + cql + calql 사전학습 → 온라인 (GitHub에서 바로 여는 판)
cql/calql은 IQL의 in-sample V(s′) 타깃을 쓰고, `td`는 59de4d9부터 prior 정책 plain TD라는 별도 method다.
**9/7 20:30 갱신**: `td`는 59de4d9에서 별도 method(prior 정책 plain TD)로 부활했고 사전학습 `td_can_s{1,2,3}.pt`가 이미 있다. 14번은 `METHODS="td cql calql"`(9 run, 150k)이고 **td는 `variant=td`로 띄운다**(이전 판의 `variant=cql`은 `check_pretrain_meta`가 method≠variant로 거부해 시작 즉시 죽는다).
이미 돌던 사전학습이 있으면 먼저 `!pkill -f offline_pretrain.py`.
20:32 감사에서 사전학습 9개는 완료, 온라인 9개는 미생성으로 확인됐다. 현재 재개 순서: 환경 복원 → 13 확인 → 14 → 15 → 12 keepalive. 11은 사전학습을 다시 만들 때만 쓴다.

## 0. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJ = '/content/drive/MyDrive/dsrl_project'
for d in ['ckpt', 'logs', 'cfg_backup', 'dppo_log']:
    os.makedirs(f'{PROJ}/{d}', exist_ok=True)
print('project dir:', PROJ)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. condacolab (실행 뒤 런타임이 자동 재시작된다. 재시작되면 2번부터)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()   # 여기서 커널 재시작

## 2. 재시작 후: Drive 다시 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/dsrl_project'
print('ok')

## 3. 저장소 클론 (브랜치 o2o). 두 서브모듈 폴더가 비어 있으면 다음 셀로 다시 받기

In [ ]:
%%bash
git config --global url."https://github.com/".insteadOf "git@github.com:"

cd /content
rm -rf dsrl
git clone --recurse-submodules -b o2o https://github.com/msp0617/dsrl.git
cd dsrl

git log --oneline -1
echo "=== dppo ==="
ls dppo | head -3
echo "=== stable-baselines3 ==="
ls stable-baselines3 | head -3

In [ ]:
%%bash
cd /content/dsrl
git submodule sync --recursive
git submodule update --init --recursive
ls dppo | head

## 4. conda 환경 복원 (Drive 캐시 `env_cache/dsrl_env.tar.gz`, 3~5분)
캐시가 없으면 v3 노트북의 4~5절(설치, 15분)을 대신 돌리고 5b 저장 셀로 캐시를 만들어 둔다.

In [ ]:
%%bash
# 복원: 새 VM에서 4~5번 대신. 1번(condacolab), 2번(Drive), 3번(클론) 뒤에 실행.
set -e
CACHE=/content/drive/MyDrive/dsrl_project/env_cache
mkdir -p /usr/local/envs
cd /usr/local/envs
rm -rf dsrl
tar -xzf $CACHE/dsrl_env.tar.gz
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python - <<'PY'
import torch, robomimic, robosuite, mujoco, stable_baselines3
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| cap", torch.cuda.get_device_capability(0))
x = torch.randn(256, 256, device="cuda"); print("matmul ok", (x @ x).sum().item() != 0)
print("robomimic", robomimic.__version__, "robosuite", robosuite.__version__, "mujoco", mujoco.__version__)
PY
echo "restored; continue from section 6"

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python /usr/local/envs/dsrl/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py

## 5. π_dp 체크포인트 (Can). Drive `dppo_log/`에서 config가 기대하는 상대경로로 복사

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
mkdir -p /content/dsrl/dppo/log

if [ -z "$(ls -A $PROJ/dppo_log 2>/dev/null)" ]; then
  echo "--- 첫 다운로드 ---"
  cd /content/dsrl/dppo/log
  gdown --folder https://drive.google.com/drive/folders/1kzC49RRFOE7aTnJh_7OvJ1K5XaDmtuh1
  cp -r /content/dsrl/dppo/log/. $PROJ/dppo_log/
else
  echo "--- Drive에서 복사 ---"
  cp -r $PROJ/dppo_log/. /content/dsrl/dppo/log/
fi
find /content/dsrl/dppo/log -maxdepth 3 | head -40

In [ ]:
%%bash
set -e

RUNTIME=/content/dsrl/dppo/log
DRIVE=/content/drive/MyDrive/dsrl_project/dppo_log

CKPT_REL=robomimic-pretrain/can/can_pre_diffusion_mlp_ta4_td20/2024-06-28_13-29-54/checkpoint/state_5000.pt
NORM_REL=robomimic/can/normalization.npz

mkdir -p "$RUNTIME" "$DRIVE"

CKPT_SRC=$(find "$RUNTIME" "$DRIVE" \
  -type f -path "*/$CKPT_REL" -print -quit 2>/dev/null || true)

NORM_SRC=$(find "$RUNTIME" "$DRIVE" \
  -type f -path "*/$NORM_REL" -print -quit 2>/dev/null || true)

echo "checkpoint: ${CKPT_SRC:-NOT_FOUND}"
echo "normalization: ${NORM_SRC:-NOT_FOUND}"

if [[ -z "$CKPT_SRC" || -z "$NORM_SRC" ]]; then
  echo "기존 다운로드에서 파일을 찾지 못했습니다."
  exit 2
fi

CKPT_DST="$RUNTIME/$CKPT_REL"
NORM_DST="$RUNTIME/$NORM_REL"

mkdir -p "$(dirname "$CKPT_DST")" "$(dirname "$NORM_DST")"

[[ "$CKPT_SRC" == "$CKPT_DST" ]] || cp -f "$CKPT_SRC" "$CKPT_DST"
[[ "$NORM_SRC" == "$NORM_DST" ]] || cp -f "$NORM_SRC" "$NORM_DST"

# 다음 세션을 위해 Drive에도 정확한 구조로 저장
mkdir -p "$DRIVE/$(dirname "$CKPT_REL")"
mkdir -p "$DRIVE/$(dirname "$NORM_REL")"
cp -f "$CKPT_DST" "$DRIVE/$CKPT_REL"
cp -f "$NORM_DST" "$DRIVE/$NORM_REL"

echo "=== READY ==="
ls -lh "$CKPT_DST" "$NORM_DST"

## 6. 환경 변수

In [ ]:
%%bash
cat > /content/env.sh <<'EOS'
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export WANDB_MODE=disabled
EOS

cat /content/env.sh

## 7. 실행 헬퍼 `run_bash`

In [ ]:
# Colab의 %%bash 는 명령이 끝나야 출력을 보여준다. 긴 학습은 이 헬퍼로 돌려서
# 한 줄씩 바로 보이게 하고, 같은 내용을 Drive의 로그 파일에도 남긴다.
import subprocess, sys

NOISE = ("Gym has been unmaintained", "Please upgrade to Gymnasium", "See the migration guide")

def run_bash(script, log_path=None):
    prefix = (
        "source /usr/local/etc/profile.d/conda.sh && conda activate dsrl\n"
        "source /content/env.sh\n"
        "cd /content/dsrl\n"
        "export HYDRA_FULL_ERROR=1 PYTHONUNBUFFERED=1\n"
    )
    log = open(log_path, "a") if log_path else None
    p = subprocess.Popen(["bash", "-c", prefix + script], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if log:
            log.write(line); log.flush()
        if not line.startswith(NOISE):
            print(line, end="", flush=True)
    p.wait()
    if log:
        log.close()
    print(f"\n[exit {p.returncode}]")
    return p.returncode

PROJ = "/content/drive/MyDrive/dsrl_project"
print("run_bash ready")

## 8. site-packages 패치

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python /content/dsrl/colab/patch_env.py

## 9. (VM2: 건너뜀) Can 청크 npz는 VM1이 15:40에 `returns` 포함으로 이미 만들었다
아래 확인 셀만 실행. `returns` 키와 gamma 0.99가 보이면 다음으로. 없으면 원본 Can 노트북(`dsrl_colab_calql_axisA.ipynb`) 9번 셀을 실행.

In [ ]:
import numpy as np
d = np.load('/content/drive/MyDrive/dsrl_project/offline/can_train_offline.npz'); r = d['returns']
print(sorted(d.files)); print('returns min %.1f mean %.1f  gamma %s  rows %d' % (r.min(), r.mean(), float(d['returns_gamma']), r.shape[0]))

## 10. 스모크 — td / cql / calql 각 200 step + 증류 100 step
`[done]`까지 가는지, `calql`의 `floored_frac`이 0과 1 사이인지. 마지막 줄의 소요 시간 × 250 ≈ 본 사전학습 1개(50k + 25k) 시간.

In [ ]:
run_bash(r'''
PROJ=/content/drive/MyDrive/dsrl_project
D="--config-path=cfg/robomimic --config-name=dsrl_can.yaml offline_data_path=$PROJ/offline/can_train_offline.npz log_dir=$PROJ/logs"
S="pretrain.steps=200 pretrain.distill_steps=100 pretrain.log_every=50 seed=0"
for M in td cql calql; do
  case $M in td) A="pretrain.method=td";; cql) A="pretrain.method=cql";; calql) A="pretrain.method=calql";; esac
  T0=$(date +%s)
  python offline_pretrain.py $D $S $A pretrain.out_path=$PROJ/logs/pretrain/smoke_$M.pt 2>&1 \
    | grep "^\[td\]\|^\[cql\]\|^\[calql\]\|^\[distill\]\|^\[done\]\|Error\|Traceback\|refuse\|ValueError"
  echo "== $M: $(( $(date +%s) - T0 )) s for 200+100 steps (x250 for 50k+25k)"
  rm -f $PROJ/logs/pretrain/smoke_$M.pt $PROJ/logs/pretrain/smoke_${M}_log.csv
done
''')

## 11. 사전학습 본 실행 — `METHODS`로 VM을 나눌 수 있다 (예: VM A `td cql`, VM B `calql`), 각 method는 seed 1~3 병렬
결과 `$PROJ/logs/pretrain/{td,cql,calql}_can_s{1,2,3}.pt` + `_log.csv`, 진행 `$PROJ/logs/pretrain_cql_<method>_s<seed>.out`. 띄운 뒤 곧바로 12번 keepalive.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && source /content/env.sh && cd /content/dsrl
git pull origin o2o | tail -n 1
METHODS="td cql calql"
SEEDS="1 2 3"
ALPHA=5.0
PROJ=/content/drive/MyDrive/dsrl_project; mkdir -p $PROJ/logs/pretrain
D="--config-path=cfg/robomimic --config-name=dsrl_can.yaml offline_data_path=$PROJ/offline/can_train_offline.npz log_dir=$PROJ/logs"
for M in $METHODS; do for SEED in $SEEDS; do
  case $M in
    td)    A="pretrain.method=td";;   # 59de4d9: prior 정책 plain TD (벌점 없음), out_path 기본값 td_can_s$SEED.pt
    cql)   A="pretrain.method=cql pretrain.cql_alpha=$ALPHA";;
    calql) A="pretrain.method=calql pretrain.cql_alpha=$ALPHA";;
  esac
  nohup python offline_pretrain.py $D seed=$SEED $A > $PROJ/logs/pretrain_cql_${M}_s${SEED}.out 2>&1 &
  echo "started $M seed $SEED (pid $!)"
done; done

## 12. keepalive (사전학습·온라인 공용). 10분마다 한 줄, 프로세스가 없으면 런타임을 스스로 반납한다
다른 셀을 돌려야 하면 이 셀 정지 → 셀 실행 → 다시 실행. 반납 없이 붙들어 두려면 `runtime.unassign()` 줄을 지운다.

In [ ]:
import subprocess, time
PROJ = '/content/drive/MyDrive/dsrl_project'
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
while True:
    running = sh("ps aux | grep '[o]ffline_pretrain.py\\|[t]rain_dsrl.py' | grep -o 'exp_id=[a-z_0-9]*\\|pretrain.method=[a-z]*\\|seed=[0-9]*' | tr '\\n' ' '")
    ram = sh("free -g | awk 'NR==2{print $3\"/\"$2}'")
    prog = sh("for f in $(ls -t %s/logs/pretrain_cql_*.out %s/logs/can_*.out 2>/dev/null | head -n 12); do "
              "n=$(basename $f .out); l=$(grep '^\\[td\\]\\|^\\[cql\\]\\|^\\[calql\\]\\|^\\[distill\\]\\|\\[eval\\]\\|\\[done\\]' $f | tail -n 1 | cut -c1-70); "
              "echo -n \"$n: $l | \"; done" % (PROJ, PROJ))
    print(time.strftime('%H:%M'), 'ram', ram, '|', running or '(none running)', '|', prog, flush=True)
    if not running:
        print('all done -> unassigning runtime', flush=True)
        from google.colab import runtime
        runtime.unassign()
        break
    time.sleep(600)

## 13. 사전학습 확인
읽는 법: `cql`에서 `q_ood_mean`이 `q_mean`을 끌고 같이 음수로 폭주하면 동료 ③의 눈금 붕괴(→ `cql_alpha`를 낮춰 재실행). `calql`은 `floored_frac`이 0.2~0.8 사이에서 안정돼야 정상. `td`는 벌점 0이므로 `q_ood_mean ≈ q_mean`.

In [ ]:
%%bash
PROJ=/content/drive/MyDrive/dsrl_project
ls -lh $PROJ/logs/pretrain/{td,cql,calql}_can_s*.pt 2>/dev/null
for F in $PROJ/logs/pretrain_cql_*.out; do echo "== $(basename $F): $(grep '^\[done\]\|Error\|Traceback' $F | tail -n 1)"; done
for M in td cql calql; do for S in 1 2 3; do
  F=$PROJ/logs/pretrain/${M}_can_s${S}_log.csv; test -f $F && echo "-- $M s$S: $(grep -v distill $F | tail -n 1 | cut -d, -f2,3,5,9,11,12,13)"
done; done
echo "(columns: phase, step, critic_loss, q_mean, penalty, q_ood_mean, floored_frac)"

## 14. 온라인 — `METHODS` × seed 1~3, 150k, 5k 격자. 두 VM으로 나누면(예: `td cql` / `calql`) run당 속도가 오른다
축 A 통제: `offline_mix.mode=none load_offline_data=False`(기본값과 같음), actor·α 로드 없음. `td`는 `variant=td`에 `td_can_sN.pt`(method==variant 검사). 띄운 뒤 12번 keepalive를 다시 실행.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && source /content/env.sh && cd /content/dsrl
git pull origin o2o | tail -n 1
METHODS="td cql calql"
SEEDS="1 2 3"
STEPS=150000
PROJ=/content/drive/MyDrive/dsrl_project
CFG="--config-path=cfg/robomimic --config-name=dsrl_can.yaml"
COMMON="log_dir=$PROJ/logs train.total_env_steps=$STEPS offline_mix.mode=none load_offline_data=False"
launch () { EXP=$1; shift; nohup python train_dsrl.py $CFG exp_id=$EXP "$@" $COMMON > $PROJ/logs/$EXP.out 2>&1 & echo "started $EXP (pid $!)"; }
for M in $METHODS; do for S in $SEEDS; do
  launch can_${M}_s$S seed=$S variant=$M pretrain_path=$PROJ/logs/pretrain/${M}_can_s$S.pt
done; done

## 15. 온라인 확인 (3~5분 뒤). 기대: run마다 `[pretrain] td|cql|calql: loaded critic, critic_target, critic_noise`(actor 없음)와 `[eval] env_steps=0`

In [ ]:
%%bash
PROJ=/content/drive/MyDrive/dsrl_project
echo "processes: $(ps aux | grep -c '[t]rain_dsrl.py') (run 수 x 2)"
for F in $(ls -t $PROJ/logs/can_{td,cql,calql}_s*.out 2>/dev/null); do
  echo "== $(basename $F .out): $(grep '\[pretrain\]\|\[eval\]\|Error\|Traceback' $F | tail -n 1 | cut -c1-110)"
done
free -g | head -2

## 16. (다른 VM에서) `calql_t12i` × 3 — 보정이 살아남는 온도
온라인 α가 1.0에서 시작하면 Cal-QL의 눈금은 25k 안에 엔트로피 보너스에 덮인다(HANDOFF 15절 iql: Q_W −145 → +70). 같은 `calql_can_sN.pt`에 `train.ent_coef=auto_0.3 train.target_ent=12`를 붙인다. VM1(Square 배치)이 반납된 뒤 새 VM에서 0~8 → 이 셀.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && source /content/env.sh && cd /content/dsrl
git pull origin o2o | tail -n 1
PROJ=/content/drive/MyDrive/dsrl_project
CFG="--config-path=cfg/robomimic --config-name=dsrl_can.yaml"
COMMON="log_dir=$PROJ/logs train.total_env_steps=150000 offline_mix.mode=none load_offline_data=False train.ent_coef=auto_0.3 train.target_ent=12"
launch () { EXP=$1; shift; nohup python train_dsrl.py $CFG exp_id=$EXP "$@" $COMMON > $PROJ/logs/$EXP.out 2>&1 & echo "started $EXP (pid $!)"; }
for S in 1 2 3; do
  launch can_calql_t12i_s$S seed=$S variant=calql pretrain_path=$PROJ/logs/pretrain/calql_can_s$S.pt
done

## 17. 결과 zip (CPU 런타임에서도 됨: 0번 Drive 마운트 → 이 셀)
로컬에서 `~/Downloads/logs/`에 풀고 `python scripts/plot_results.py --logs ~/Downloads/logs --out <figs> --axes "critic=baseline,td,iql,cql,calql,calql_t12i,warmupc"` → `summary.csv`(T50/T80/AUC/final).

In [ ]:
%%bash
cd /content/drive/MyDrive/dsrl_project
rm -f csv_bundle.zip
zip -qr csv_bundle.zip logs -i "logs/*/eval_log.csv" "logs/*/train_log.csv" "logs/*.csv" "logs/pretrain/*_log.csv"
ls -lh csv_bundle.zip